# spore — Library Overview

**spore** is a simulation framework for sampling neutrino events from astrophysical sources through a parameterized detector response. It supports:

- **Point sources** (steady-state or transient), **diffuse/extended sources**, and **dark-matter Galactic-halo sources**
- **Multi-detector** analyses with independent response functions
- Sampling via Metropolis–Hastings MCMC over `(sin δ, RA, log E)` space
- Detector responses loaded from HDF5 files or TOML + CSV configurations, including the IceCube 10-year data-release IRFs
- Event objects that carry **physical units** via `pint`

This notebook walks through all major features from detector construction to event I/O.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

# spore imports
from spore.conventions import SkyCoordinate, units
from spore.conventions.units import ureg           # shared pint UnitRegistry
from spore.detector import Detector
from spore.source import (
    PointSource,
    ExtendedSource,
    NFWProfile, GalacticHaloAnnihilationSource, SoftSpectrum, LineSpectrum,
)
from spore.source.flux.flux import Flux
from spore.source.flux.distributions.power_laws import PowerLaw
from spore.physics import neutrinos
from spore.event_sampling import (
    PointSourceEventSampler,
    ExtendedSourceEventSampler,
    GalacticHaloEventSampler,
    MultiDetectorPointSourceSampler,
)
from spore.event_sampling.io import write_events, read_events

# Paths (relative to notebook; adjust if running from a different working directory)
REPO      = Path("..")
RESOURCES = REPO / "resources"
SCRATCH   = REPO / "scratch"

EXAMPLE_RESPONSE  = str(RESOURCES / "example_detector_response.h5")
COMBINED_FLUX_H5  = str(SCRATCH  / "combined_flux.h5")

---
## 1. Detector construction

A `Detector` bundles:

| Attribute | Description |
|---|---|
| `location` | `EarthCoordinate(latitude, longitude)` in radians |
| `depth` | Instrumented depth in metres |
| `medium` | `"Ice"` or `"Water"` |
| `response` | `DetectorResponse` (effective area, PSF, energy resolution) |

There are three equivalent construction routes.

In [ ]:
# --- Route 1: from a configuration dictionary ---
det_icecube = Detector.from_config({
    "properties": {
        "latitude":  -90.0,   # degrees
        "longitude":   0.0,
        "depth":    1945,     # metres
        "medium":  "Ice",
    },
    "response": {"detector_response_file": EXAMPLE_RESPONSE},
})

# --- Route 2: from a single TOML file (preferred for reproducibility) ---
# resources/configs/icecube_properties.toml references ps10yr_response.toml
# via a relative path, which in turn points to the data-release IRF directory.
# Uncomment once the data release is available locally (see Section 6).
#
# det_icecube = Detector.from_toml(str(RESOURCES / "configs" / "icecube_properties.toml"))

# --- Route 3: Mediterranean detector (KM3NeT ARCA-like) ---
det_km3net = Detector.from_config({
    "properties": {
        "latitude":  36.3,
        "longitude": 16.1,
        "depth":    3500,
        "medium":  "Water",
    },
    "response": {"detector_response_file": EXAMPLE_RESPONSE},
})

print("IceCube latitude:  ", np.degrees(det_icecube.location.latitude),  "deg")
print("KM3NeT latitude:   ", np.degrees(det_km3net.location.latitude),   "deg")
print("Available morphologies:", det_icecube.response.available_morphologies)

---
## 2. Detector response formats

spore supports two storage formats for detector responses.

### 2a. HDF5 format

The simplest format is a self-contained HDF5 file with one group per morphology (`track`, `cascade`). Each group may contain:

```
/{morphology}_effective_area/
    energies             (n_E,)              log₁₀(E/GeV) grid
    zeniths              (n_zen,)             zenith angle grid [rad]
    tabulated_values     (n_E, n_zen)         A_eff [cm²]
    lower_bounds         (2, 2)               polygon mask corners
    upper_bounds         (1, 1)

/{morphology}_angular_response/
    energies             (n_E,)              energy nodes [eV]
    us                   (n_u,)              uniform [0, 1] knots
    inv_cdfs             (n_E, n_u)          inverse CDF: u → opening angle [rad]

/{morphology}_energy_resolution/
    us                   (n_u,)              uniform knots
    inv_cdf              (n_u,)              inverse CDF: u → log(E_reco / E_true)
```

Sampling energy and angles independently from these 1-D inverse CDFs is fast and exact.

In [ ]:
import h5py

print("Contents of example_detector_response.h5:")
with h5py.File(EXAMPLE_RESPONSE, "r") as f:
    def _show(name, obj):
        if hasattr(obj, "shape"):
            print(f"  {name}: shape={obj.shape}")
        else:
            print(f"  {name}/")
    f.visititems(_show)

### 2b. TOML + CSV format

For more flexibility, a TOML file points to CSV files for the effective area and describes the PSF analytically:

```toml
# resources/configs/gfu_response.toml
[meta]
name   = "IceCube GFU"
sample = "GFU"

[track.effective_area]
# CSV columns: log10_e_gev  sindec_min  sindec_max  aeff_cm2
file = "gfu_effective_area.csv"

[track.psf]
form          = "rayleigh_power_law"
amplitude_deg = 1.0      # median PSF at pivot energy
pivot_gev     = 1000.0   # 1 TeV
index         = 0.5      # median PSF ∝ E^{-index}
```

Paths in the TOML are resolved in order:
1. Relative to the TOML file itself
2. Relative to the current working directory
3. As an absolute path

### 2c. IceCube data-release IRFs

The 10-year point-source data release (arXiv:2101.09836) ships per-season IRF CSV files. spore can average these into a single response:

```toml
# resources/configs/ps10yr_response.toml
[track.effective_area]
type    = "dataverse_csv"
irf_dir = "/path/to/dataverse_files/irfs"
seasons = ["IC86_I", "IC86_II", "IC86_III", "IC86_IV", "IC86_V", "IC86_VI", "IC86_VII"]

[track.smearing]
type    = "dataverse_csv"
irf_dir = "/path/to/dataverse_files/irfs"
seasons = ["IC86_I", "IC86_II", ...]
```

The `smearing` block loads a 5-D histogram `P(E_reco, PSF, AngErr | E_true, dec)` and builds joint inverse-CDF samplers per `(E_true, dec)` cell. This captures correlations between energy reconstruction and angular resolution.

In [ ]:
# Visualise the example effective area vs zenith and energy
from spore.detector.detector_response.detector_response import DetectorResponse

response = DetectorResponse.from_config({"detector_response_file": EXAMPLE_RESPONSE})
aeff     = response.effective_area["track"]

log10e = np.linspace(2, 6, 80)     # log10(E / GeV)
es_gev = 10 ** log10e
zeniths_deg = [91, 100, 120, 150]  # upgoing for IceCube (> 90°)

fig, ax = plt.subplots(figsize=(7, 4))
for zen_deg in zeniths_deg:
    zen = np.radians(zen_deg)
    aeff_vals = np.array([aeff(zen, e * units.GeV) for e in es_gev])
    ax.plot(es_gev, aeff_vals / 1e4, label=f"ζ = {zen_deg}°")  # cm² → m²

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"$E$ [GeV]")
ax.set_ylabel(r"$A_{\rm eff}$ [m$^2$]")
ax.set_title("Track effective area vs energy (example response)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## 3. Sources

### 3a. PointSource

A `PointSource` is defined by a sky position and a `Flux` object. The flux is evaluated per neutrino species and energy.

In [ ]:
# From a TOML file — the easiest route
src_toml = PointSource.from_toml(str(RESOURCES / "configs" / "powerlaw_source_example.toml"))
print("Source from TOML:")
print(f"  RA  = {np.degrees(src_toml.location.right_ascension):.1f} deg")
print(f"  Dec = {np.degrees(src_toml.location.declination):.1f} deg")

# From code — more control over individual parameters
emin  = 1e2 * units.GeV
emax  = 1e6 * units.GeV
pivot = 1e5 * units.GeV   # 100 TeV
pl    = PowerLaw(gamma=2.0, emin=emin, emax=emax, pivot=pivot)

# norm_per_species is in spore natural units: eV^-1 cm^-2 eV^-2 = eV^-1 eV^2 / (cm^2 eV)
# For a total (nu + nubar) track flux of 5e-11 TeV^-1 cm^-2 s^-1 at 1 TeV:
#   5e-14 GeV^-1 cm^-2 s^-1  →  divide by 2 for per-species
norm = 2.5e-14 / units.GeV / units.cm**2 / units.sec
flux = Flux(
    normalizations={nu: norm for nu in neutrinos},
    distributions={nu: pl   for nu in neutrinos},
)
src_code = PointSource(
    flux=flux,
    location=SkyCoordinate(np.radians(5.0), np.radians(50.0)),  # dec=5°, RA=50°
)

# Both sources evaluate identically
print(f"\nFlux at 1 TeV (NuMu, from TOML): {src_toml(neutrinos[2], 1e3 * units.GeV):.3e}")
print(f"Flux at 1 TeV (NuMu, from code): {src_code(neutrinos[2], 1e3 * units.GeV):.3e}")

### 3b. GalacticHaloAnnihilationSource

The Galactic halo source models dark matter annihilation. The differential flux per solid angle is:

$$\frac{d\Phi}{dE\,d\Omega} = \frac{\langle\sigma v\rangle}{8\pi m_\chi^2} \frac{dN}{dE}(E)\, J(\mathrm{RA}, \delta)$$

where $J$ is the line-of-sight integral of the squared DM density (the J-factor). Because the Galactic Centre is not at the celestial pole, $J$ varies across the sky — this 2-D dependence is handled correctly by `GalacticHaloEventSampler`.

Three spectral models are built in:
- **`LineSpectrum`**: monochromatic $\chi\chi \to \nu\bar\nu$
- **`BoxSpectrum`**: flat distribution up to $m_\chi$
- **`SoftSpectrum`**: power-law with kinematic cutoff $\propto (E/m_\chi)^a (1 - E/m_\chi)^b$

In [ ]:
M_CHI_GEV = 1e4            # 10 TeV DM mass
SIGMA_V   = 1.23e-24       # cm³/s

profile  = NFWProfile(rho_s=0.3, r_s=20.0, r_sun=8.5)  # canonical Milky Way
spectrum = SoftSpectrum(M_CHI_GEV * 1e9, a=1.5, b=1.0, n_nu=3.0)  # bb-bar proxy

src_halo = GalacticHaloAnnihilationSource(
    profile     = profile,
    spectrum    = spectrum,
    m_chi_gev   = M_CHI_GEV,
    sigma_v_cm3s= SIGMA_V,
)

# Monochromatic (line) source — drops energy dimension from MCMC
src_line = GalacticHaloAnnihilationSource(
    profile     = profile,
    spectrum    = LineSpectrum(M_CHI_GEV * 1e9),
    m_chi_gev   = M_CHI_GEV,
    sigma_v_cm3s= SIGMA_V,
)
print(f"SoftSpectrum source is_monochromatic: {src_halo.is_monochromatic}")
print(f"LineSpectrum source is_monochromatic: {src_line.is_monochromatic}")
print(f"Line source injection energy: {src_line.E_0_eV / units.TeV:.1f} TeV")

---
## 4. Point source sampling

`PointSourceEventSampler` uses the effective area at the source's zenith angle (which varies with time) and the source flux to build a 1-D energy CDF. Sampling then draws energies from that CDF and smears the direction using the PSF.

In [ ]:
T_OBS = 365.25 * units.day   # 1 year in natural units (eV⁻¹)
T_MJD = 60355.0              # reference epoch

sampler_ps = PointSourceEventSampler(det_icecube, src_toml)

# expected_events returns the mean Poisson rate for a given observation window
lam_track   = sampler_ps.expected_events("track",   T_OBS)
lam_cascade = sampler_ps.expected_events("cascade", T_OBS)
print(f"Expected track events (1 yr):   {lam_track:.2f}")
print(f"Expected cascade events (1 yr): {lam_cascade:.2f}")

# Mode 1: fixed number of events (pseudo-experiment)
events_fixed = sampler_ps.sample_events("track", nevent=50)
print(f"\nSampled (fixed nevent=50): {len(events_fixed)} track events")

# Mode 2: Poisson-draw from observation window
events_poisson = sampler_ps.sample_events("track", deltat=T_OBS, t=T_MJD)
print(f"Sampled (Poisson, 1 yr):   {len(events_poisson)} track events")

### 4a. Working with Event objects

Each `Event` carries physical quantities as `pint.Quantity` objects, so units are always explicit. `to_dict()` returns plain floats (magnitudes in natural units) for downstream analysis.

In [ ]:
ev = events_fixed[0]

print("Event fields:")
print(f"  true_energy  = {ev.true_energy}")
print(f"  reco_energy  = {ev.reco_energy}")
print(f"  ang_err      = {ev.ang_err}")
print(f"  true_dec     = {np.degrees(ev.true_direction.declination):.2f} deg")
print(f"  morphology   = {ev.morphology}")

# Convert to convenient units using pint
print(f"\n  true energy in TeV: {ev.true_energy.to('TeV')}")
print(f"  ang_err in degrees: {ev.ang_err.to('deg')}")

# Plain-float extraction for array operations
true_e_tev = np.array([e.true_energy.to("TeV").magnitude for e in events_fixed])
reco_e_tev = np.array([e.reco_energy.to("TeV").magnitude for e in events_fixed])
ang_err_deg= np.array([e.ang_err.to("deg").magnitude    for e in events_fixed])
true_decs  = np.degrees([e.true_direction.declination     for e in events_fixed])
reco_decs  = np.degrees([e.reco_direction.declination     for e in events_fixed])
true_ras   = np.degrees([e.true_direction.right_ascension for e in events_fixed])
reco_ras   = np.degrees([e.reco_direction.right_ascension for e in events_fixed])

# to_dict() returns magnitudes in eV (spore natural units)
print("\nto_dict() keys:", list(ev.to_dict().keys()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Sky map
ax = axes[0]
sc = ax.scatter(reco_ras, reco_decs, c=np.log10(reco_e_tev), cmap="plasma", s=20, alpha=0.8)
ax.scatter(np.degrees(src_toml.location.right_ascension),
           np.degrees(src_toml.location.declination),
           marker="*", s=250, color="gold", zorder=5, label="True source")
plt.colorbar(sc, ax=ax, label=r"$\log_{10}(E_\mathrm{reco}\,/\,\mathrm{TeV})$")
ax.set_xlabel("RA [deg]")
ax.set_ylabel("Dec [deg]")
ax.set_title("Reconstructed directions (nevent=50)")
ax.legend(fontsize=8)

# Energy comparison
ax = axes[1]
bins = np.logspace(np.log10(true_e_tev.min() * 0.5), np.log10(true_e_tev.max() * 2), 20)
ax.hist(true_e_tev, bins=bins, histtype="step", lw=1.5, label="True energy")
ax.hist(reco_e_tev, bins=bins, histtype="step", lw=1.5, ls="--", label="Reco energy")
ax.set_xscale("log")
ax.set_xlabel(r"$E$ [TeV]")
ax.set_ylabel("Events / bin")
ax.set_title("Energy reconstruction")
ax.legend()

plt.tight_layout()
plt.show()

---
## 5. Galactic halo (dark matter) sampling

`GalacticHaloEventSampler` builds a 3-D or 2-D (monochromatic) grid of `A_eff × dΦ/dE dΩ` over `(sin δ, RA, log E)` and runs MCMC to sample from that target distribution. The J-factor variation across the sky is fully accounted for.

Grid construction involves:
1. Vectorized `psi_from_radec_grid` for all `(dec, RA)` pairs (single `astropy.SkyCoord` batch)
2. Per-direction `scipy.quad` J-factor integrals (unavoidable)
3. Single vectorized effective-area call over the flattened grid

> **Note:** For demonstration we use a very coarse grid (`n_dec=10, n_ra=11`). Production runs typically use 50–100 per axis.

In [ ]:
from spore.source.jfactor import psi_from_radec, j_factor

T_OBS_HALO = 10.0 * 365.25 * units.day   # 10 years

sampler_halo = GalacticHaloEventSampler(
    det_km3net,         # Mediterranean — GC accessible as upgoing source
    src_halo,
    burnin = 5_000,
    n_dec  = 10,
    n_ra   = 11,
    n_e    = 12,
)

print(f"Expected track events (10 yr):   {sampler_halo.expected_events('track',   T_OBS_HALO):.1f}")
print(f"Expected cascade events (10 yr): {sampler_halo.expected_events('cascade', T_OBS_HALO):.1f}")

events_halo = sampler_halo.sample_events("track", nevent=200, t=T_MJD)
print(f"\nSampled {len(events_halo)} track events")

In [ ]:
# Extract quantities
halo_true_decs  = np.degrees([e.true_direction.declination     for e in events_halo])
halo_true_ras   = np.degrees([e.true_direction.right_ascension for e in events_halo])
halo_reco_decs  = np.degrees([e.reco_direction.declination     for e in events_halo])
halo_reco_ras   = np.degrees([e.reco_direction.right_ascension for e in events_halo])
halo_true_e_tev = np.array([e.true_energy.to("TeV").magnitude for e in events_halo])
halo_reco_e_tev = np.array([e.reco_energy.to("TeV").magnitude for e in events_halo])

# Angle from Galactic Centre for each true direction
psis_deg = np.degrees([
    psi_from_radec(np.radians(ra), np.radians(dec))
    for ra, dec in zip(halo_true_ras, halo_true_decs)
])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle(
    f"NFW halo — $m_\\chi$ = {M_CHI_GEV/1e3:.0f} TeV, "
    f"$\\langle\\sigma v\\rangle$ = {SIGMA_V:.0e} cm$^3$/s (KM3NeT location)",
    fontsize=12,
)

# Sky map
ax = axes[0]
sc = ax.scatter(halo_reco_ras, halo_reco_decs,
                c=np.log10(halo_reco_e_tev), cmap="plasma", s=6, alpha=0.7)
ax.scatter([266.4], [-28.9], s=200, marker="*", color="lime",
           zorder=5, label="Galactic Centre")
plt.colorbar(sc, ax=ax, label=r"$\log_{10}(E_\mathrm{reco}/\mathrm{TeV})$")
ax.set_xlabel("RA [deg]")
ax.set_ylabel("Dec [deg]")
ax.set_title("Sky map (reco directions)")
ax.legend(fontsize=8)

# Angular distribution vs NFW theory
ax = axes[1]
psi_bins = np.linspace(0, 180, 31)
counts, edges = np.histogram(psis_deg, bins=psi_bins)
cents = 0.5 * (edges[:-1] + edges[1:])
pdf   = counts / (counts.sum() * (psi_bins[1] - psi_bins[0]))
ax.step(cents, pdf, where="mid", lw=1.5, label="Sampled events")

psi_th = np.linspace(0.1, 179.9, 300)
j_th   = np.array([j_factor(profile, np.radians(p)) for p in psi_th])
pdf_th = j_th * np.sin(np.radians(psi_th))
pdf_th /= np.trapz(pdf_th, psi_th)
ax.plot(psi_th, pdf_th, color="tomato", lw=1.5, label=r"NFW $J(\psi)\sin\psi$")
ax.set_xlabel(r"Angle from GC [deg]")
ax.set_ylabel(r"Probability density [deg$^{-1}$]")
ax.set_title("Angular distribution (true directions)")
ax.legend(fontsize=8)

# Energy spectrum
ax = axes[2]
bins_e = np.logspace(np.log10(halo_true_e_tev.min() * 0.5),
                     np.log10(M_CHI_GEV / 1e3 * 1.5), 25)
ax.hist(halo_true_e_tev, bins=bins_e, histtype="step", lw=1.5, label="True")
ax.hist(halo_reco_e_tev, bins=bins_e, histtype="step", lw=1.5, ls="--", label="Reco")
ax.axvline(M_CHI_GEV / 1e3, ls=":", color="gray", label=f"$m_\\chi$ = {M_CHI_GEV/1e3:.0f} TeV")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"$E$ [TeV]")
ax.set_ylabel("Events / bin")
ax.set_title("Energy spectrum")
ax.legend()

plt.tight_layout()
plt.show()

---
## 6. Multi-detector point source sampling

`MultiDetectorPointSourceSampler` wraps multiple `PointSourceEventSampler` instances and tags each event with `detector_id`. This is the entry point for joint-likelihood analyses across detector networks.

In [ ]:
source_multidet = PointSource(
    flux=flux,
    location=SkyCoordinate(np.radians(-30.0), np.radians(83.8)),  # dec=-30°, RA=83.8°
)

sampler_multi = MultiDetectorPointSourceSampler(
    detectors=[
        (det_icecube, "IceCube"),
        (det_km3net,  "KM3NeT"),
    ],
    source=source_multidet,
)

events_multi = sampler_multi.sample_events("track", t=T_MJD, deltat=T_OBS)

by_det = defaultdict(list)
for ev in events_multi:
    by_det[ev.detector_id].append(ev)

print(f"Total events: {len(events_multi)}")
for name, evs in by_det.items():
    med_e = np.median([e.true_energy.to('TeV').magnitude for e in evs])
    print(f"  {name}: {len(evs)} events, median true E = {med_e:.2f} TeV")

# Sky map coloured by detector
fig, ax = plt.subplots(figsize=(8, 5))
colors = {"IceCube": "steelblue", "KM3NeT": "tomato"}
for name, evs in by_det.items():
    ras  = np.degrees([e.reco_direction.right_ascension for e in evs])
    decs = np.degrees([e.reco_direction.declination     for e in evs])
    ax.scatter(ras, decs, s=10, alpha=0.6, color=colors[name], label=name)

ax.scatter(
    np.degrees(source_multidet.location.right_ascension),
    np.degrees(source_multidet.location.declination),
    marker="*", s=250, color="gold", zorder=5, label="Source"
)
ax.set_xlabel("RA [deg]")
ax.set_ylabel("Dec [deg]")
ax.set_title("Multi-detector — reconstructed directions (1 year, tracks)")
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. Extended / diffuse source sampling

`ExtendedSource` takes a flux table stored in an HDF5 file. The table has shape `(6, n_dec, n_E)` indexing over neutrino species, declination bands, and energies.

`ExtendedSourceEventSampler` supports two modes:

- **Transient** (`deltat=None`): A_eff is evaluated at a single epoch; RA dependence is resolved.
- **Steady-state** (`deltat=T`): A_eff is averaged over the full diurnal cycle (correct for multi-day observations). The resulting distribution is RA-independent and directly reflects the detector's time-averaged sky coverage.

In [ ]:
# Load pre-computed atmospheric and astrophysical fluxes from combined_flux.h5.
# The file has groups: 'atmospheric', 'astrophysical', 'combined'.
# Each group has datasets: sindecs (n_dec,), energies (n_E,) [GeV], fluxes (6, n_dec, n_E)

T_OBS_EXT = 365.25 * units.day   # 1 year

src_atmo  = ExtendedSource.from_config({"flux": {"location": f"{COMBINED_FLUX_H5}:atmospheric"}})
src_astro = ExtendedSource.from_config({"flux": {"location": f"{COMBINED_FLUX_H5}:astrophysical"}})

# Steady-state mode: pass deltat to constructor to enable HA-averaging
sampler_atmo = ExtendedSourceEventSampler(
    det_icecube, src_atmo,
    deltat=T_OBS_EXT,
    n_dec=20, n_ra=20, n_e=20,
    n_time_samples=40,
)
sampler_astro = ExtendedSourceEventSampler(
    det_icecube, src_astro,
    deltat=T_OBS_EXT,
    n_dec=20, n_ra=20, n_e=20,
    n_time_samples=40,
)

print(f"Expected atmospheric events  (1 yr, track): {sampler_atmo.expected_events('track',  T_OBS_EXT):.0f}")
print(f"Expected astrophysical events (1 yr, track): {sampler_astro.expected_events('track', T_OBS_EXT):.0f}")

In [ ]:
events_atmo  = sampler_atmo.sample_events("track",  deltat=T_OBS_EXT, oversample=3)
events_astro = sampler_astro.sample_events("track", deltat=T_OBS_EXT, oversample=3)

print(f"Sampled atmospheric:   {len(events_atmo)}")
print(f"Sampled astrophysical: {len(events_astro)}")

atmo_reco_decs  = np.degrees([e.reco_direction.declination for e in events_atmo])
astro_reco_decs = np.degrees([e.reco_direction.declination for e in events_astro])
atmo_reco_e_gev  = np.array([e.reco_energy.to("GeV").magnitude for e in events_atmo])
astro_reco_e_gev = np.array([e.reco_energy.to("GeV").magnitude for e in events_astro])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle("Extended source — steady-state mode, IceCube (1 year, tracks)")

ax = axes[0]
bins_dec = np.linspace(-90, 90, 30)
ax.hist(atmo_reco_decs,  bins=bins_dec, histtype="step", lw=1.5, label="Atmospheric")
ax.hist(astro_reco_decs, bins=bins_dec, histtype="step", lw=1.5, label="Astrophysical")
ax.set_xlabel("Reconstructed declination [deg]")
ax.set_ylabel("Events / bin")
ax.set_title("Declination distribution")
ax.legend()

ax = axes[1]
bins_e = np.logspace(2, 7, 30)
ax.hist(atmo_reco_e_gev,  bins=bins_e, histtype="step", lw=1.5, label="Atmospheric")
ax.hist(astro_reco_e_gev, bins=bins_e, histtype="step", lw=1.5, label="Astrophysical")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"$E_\mathrm{reco}$ [GeV]")
ax.set_ylabel("Events / bin")
ax.set_title("Reconstructed energy spectrum")
ax.legend()

plt.tight_layout()
plt.show()

---
## 8. Comparison to the IceCube 10-year data release

This section compares spore simulations to the IceCube 10-year point-source data release (arXiv:2101.09836). The data release can be downloaded from the IceCube dataverse.

**Requirements:**
- IceCube PS 10yr data release, available at: https://icecube.wisc.edu/data-releases/2021/01/all-sky-point-source-icecube-data-years-2008-2018/
- Set `DATA_RELEASE` below to the local path of the downloaded `dataverse_files` directory.
- The detector TOML `ps10yr_response.toml` must point to the correct `irfs/` subdirectory.

The comparison focuses on the **northern sky** (sin δ > 0), where the IceCube 10-year sample is dominated by upgoing muon neutrinos and atmospheric contamination is suppressed relative to the southern sky.

In [ ]:
import os

# --- Set this to your local data release path ---
DATA_RELEASE = os.path.expanduser("~/Downloads/dataverse_files 2")
HAS_DATA_RELEASE = os.path.isdir(DATA_RELEASE)

if not HAS_DATA_RELEASE:
    print(f"Data release not found at {DATA_RELEASE!r}.")
    print("Skipping data release comparison. Download from:")
    print("https://icecube.wisc.edu/data-releases/2021/01/all-sky-point-source-icecube-data-years-2008-2018/")
else:
    print(f"Data release found: {DATA_RELEASE}")

In [ ]:
%%script echo "Skipping — requires data release" --no-raise-error
# Remove the %%script line above once DATA_RELEASE is set correctly.

# --- Compute total livetime from uptime files ---
IC86_SEASONS = ['IC86_I', 'IC86_II', 'IC86_III', 'IC86_IV', 'IC86_V', 'IC86_VI', 'IC86_VII']

total_days = 0.0
for season in IC86_SEASONS:
    uptime_file = os.path.join(DATA_RELEASE, "uptime", f"{season}_exp.csv")
    ut = np.genfromtxt(uptime_file, comments="#")
    total_days += (ut[:, 1] - ut[:, 0]).sum()

T_DR = total_days * units.day
print(f"Total IC86 livetime: {total_days:.1f} days ({total_days / 365.25:.2f} years)")

In [ ]:
%%script echo "Skipping — requires data release" --no-raise-error

# --- Load data release events (all IC86 seasons) ---
IC86_EVENT_FILES = {
    'IC86_I':   'IC86_I_exp.csv',
    'IC86_II':  'IC86_II_exp.csv',
    'IC86_III': 'IC86_III_exp.csv',
    'IC86_IV':  'IC86_IV_exp.csv',
    'IC86_V':   'IC86_V_exp.csv',
    'IC86_VI':  'IC86_VI_exp.csv',
    'IC86_VII': 'IC86_VII_exp-1.csv',
}

all_ev = np.vstack([
    np.genfromtxt(os.path.join(DATA_RELEASE, "events", f), comments="#")
    for f in IC86_EVENT_FILES.values()
])

# Columns: MJD, log10(E_reco/GeV), ang_err [deg], RA [deg], Dec [deg], ...
log10e_dr = all_ev[:, 1]
sindec_dr = np.sin(np.radians(all_ev[:, 4]))

print(f"Total data release events: {len(all_ev):,}")
print(f"North sky (sin δ > 0):    {(sindec_dr > 0).sum():,}")

In [ ]:
%%script echo "Skipping — requires data release" --no-raise-error

# --- Build IceCube detector with PS10yr response ---
# Edit ps10yr_response.toml to point irf_dir at your local data release.
det_ps10yr = Detector.from_toml(str(RESOURCES / "configs" / "icecube_properties.toml"))

# --- Build extended source samplers (steady-state mode over full livetime) ---
sampler_atmo_dr = ExtendedSourceEventSampler(
    det_ps10yr, src_atmo,
    deltat=T_DR,
    n_dec=40, n_ra=40, n_e=40,
    n_time_samples=100,
)
sampler_astro_dr = ExtendedSourceEventSampler(
    det_ps10yr, src_astro,
    deltat=T_DR,
    n_dec=40, n_ra=40, n_e=40,
    n_time_samples=100,
)

print(f"Expected atmospheric events  (north, {total_days:.0f} d): "
      f"{sampler_atmo_dr.expected_events('track', T_DR):.0f}")
print(f"Expected astrophysical events (north, {total_days:.0f} d): "
      f"{sampler_astro_dr.expected_events('track', T_DR):.0f}")

In [ ]:
%%script echo "Skipping — requires data release" --no-raise-error

events_atmo_dr  = sampler_atmo_dr.sample_events( "track", deltat=T_DR, oversample=2)
events_astro_dr = sampler_astro_dr.sample_events("track", deltat=T_DR, oversample=2)

print(f"Sampled atmospheric:   {len(events_atmo_dr):,}")
print(f"Sampled astrophysical: {len(events_astro_dr):,}")

In [ ]:
%%script echo "Skipping — requires data release" --no-raise-error

# Extract north-sky reco energies
def north_reco_e_gev(event_list):
    return np.array([
        e.reco_energy.to("GeV").magnitude
        for e in event_list
        if e.reco_direction.declination < np.pi / 2
    ])

e_atmo_dr  = north_reco_e_gev(events_atmo_dr)
e_astro_dr = north_reco_e_gev(events_astro_dr)

# Energy bins matching the data release log scale
e_bins  = np.logspace(2, 8, 40)
e_cents = 0.5 * (e_bins[:-1] + e_bins[1:])

h_atmo_dr,  _ = np.histogram(e_atmo_dr,  bins=e_bins)
h_astro_dr, _ = np.histogram(e_astro_dr, bins=e_bins)
h_data_dr,  _ = np.histogram(10 ** log10e_dr[sindec_dr > 0], bins=e_bins)

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
fig.suptitle(
    f"IceCube 10-year data release vs spore simulation\n"
    f"North sky (sin δ > 0), {total_days:.0f} days, tracks",
    fontsize=12,
)

ax = axes[0]
ax.step(e_cents, h_data_dr,                 where="mid", lw=2,   color="black",   label="IC86 data")
ax.step(e_cents, h_atmo_dr + h_astro_dr,    where="mid", lw=2,   color="steelblue",label="spore total")
ax.step(e_cents, h_atmo_dr,                 where="mid", lw=1.5, ls="--", color="cornflowerblue", label="Atmospheric")
ax.step(e_cents, h_astro_dr,                where="mid", lw=1.5, ls=":",  color="tomato",        label="Astrophysical")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(1e2, 1e8)
ax.set_ylabel("Events / bin")
ax.legend(fontsize=9)

ax = axes[1]
residual = h_atmo_dr + h_astro_dr - h_data_dr
sigma    = np.sqrt(h_data_dr + h_atmo_dr + h_astro_dr)
with np.errstate(divide="ignore", invalid="ignore"):
    pull = np.where(sigma > 0, residual / sigma, 0.0)
ax.step(e_cents, pull, where="mid", lw=1.5, color="steelblue")
ax.axhline(0, ls="--", color="gray", lw=1)
ax.axhline( 1, ls=":",  color="gray", lw=0.8)
ax.axhline(-1, ls=":",  color="gray", lw=0.8)
ax.set_xlabel(r"$E_\mathrm{reco}$ [GeV]")
ax.set_ylabel(r"(spore $-$ data) / $\sqrt{N}$")
ax.set_xscale("log")
ax.set_xlim(1e2, 1e8)
ax.set_ylim(-3, 3)

plt.tight_layout()
plt.show()

---
## 9. Saving and loading events

Events are serialised to HDF5 via `write_events` / `read_events`. Multiple event lists can be stored in the same file under different group names (e.g., `"signal"`, `"background"`).

In [ ]:
import tempfile

outfile = "/tmp/spore_demo_events.h5"

# Write two event lists to separate groups
write_events(events_fixed,  outfile, "point_source")
write_events(events_halo,   outfile, "galactic_halo")

# Reload and verify
ps_loaded   = read_events(outfile, "point_source")
halo_loaded = read_events(outfile, "galactic_halo")

print(f"Reloaded {len(ps_loaded)} point-source events")
print(f"Reloaded {len(halo_loaded)} galactic-halo events")

# Energy fields survive the round-trip as plain floats (magnitudes in eV)
# and are re-wrapped as pint Quantities on load
ev0 = ps_loaded[0]
print(f"\nFirst PS event after reload:")
print(f"  true_energy = {ev0.true_energy}")
print(f"  ang_err     = {ev0.ang_err}")

---
## Summary

| Feature | Class / function |
|---|---|
| Detector from config dict | `Detector.from_config(cfg)` |
| Detector from TOML | `Detector.from_toml(path)` |
| Point source | `PointSource.from_toml(path)` or `PointSource(flux, location)` |
| Dark matter halo source | `GalacticHaloAnnihilationSource(profile, spectrum, m_chi_gev, sigma_v_cm3s)` |
| Diffuse source from H5 | `ExtendedSource.from_config({"flux": {"location": "file.h5:group"}})` |
| Point source sampling | `PointSourceEventSampler(det, src)` |
| Galactic halo sampling | `GalacticHaloEventSampler(det, src, n_dec, n_ra, n_e)` |
| Diffuse source sampling | `ExtendedSourceEventSampler(det, src, deltat=T)` |
| Multi-detector | `MultiDetectorPointSourceSampler([(det1,name1), ...], source)` |
| Expected event rate | `sampler.expected_events(morphology, deltat)` |
| Sample fixed count | `sampler.sample_events(morphology, nevent=N)` |
| Sample from livetime | `sampler.sample_events(morphology, deltat=T)` |
| Unitful event fields | `event.true_energy`, `event.reco_energy`, `event.ang_err` (pint Quantities) |
| Unit conversion | `event.true_energy.to('TeV')` |
| Save events | `write_events(events, path, group)` |
| Load events | `read_events(path, group)` |